# Anonymize a DICOM file

This example is a starting point to anonymize DICOM data.

Notes for running:
- Activate the venv for reproducible results: `source venv_cmepda/bin/activate`
- You can set the dataset root with the environment variable `CMEPDA_DATASETS`
  or edit the `DATASETS` variable in the notebook (see the DATASETS cell).
- The notebook contains optional Google Colab mount lines (commented).

It shows how to read data and replace tags: person names, patient id,
optionally remove private tags, and write the results in a new file

## Reading the dataset from Google Drive
Prior to this operation be sure to have added the shared folder to your Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls "/content/drive/My Drive/DATASETS"

FEATURES  IMAGES  README.rtf


In [3]:
DATASETS = "/content/drive/My Drive/DATASETS"

## Reading the dataset from a local folder

In [ ]:
# Set this to your local dataset folder
#DATASETS = "/Users/retico/Desktop/CMEPDA/DATASETS"

## Anonymize a single file

In [4]:
%pip install pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.9 MB/s eta 0:00:00


In [5]:
import os
import pydicom

In [6]:
filename = os.path.join(DATASETS, "IMAGES", "DICOM_Examples", "Brain_MRI", "IM67_1slice.dcm")
dataset = pydicom.dcmread(filename)

We can prints the `dataset` object to see a summary of the `FileDataset` (file meta information and top-level DICOM tags). Pixel data can be large and is not expanded by this print.

In [7]:
dataset

Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 228
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: MR Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.6.1.4.1.9590.100.1.2.4365715539284982230698868930013138318
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.6.1.4.1.9590.100.1.3.100.9.4
(0002,0013) Implementation Version Name         SH: 'MATLAB IPT 9.4'
(0002,0016) Source Application Entity Title     AE: 'SMARDicomSCP'
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'OTHER']
(0008,0016) SOP Class UID                       UI: MR Image Storage
(0008,0018) SOP Instance UID                    UI: 1.3.6.1.4.1.9590.100.1.2.4

Specific elements can be accessed, e.g. `WindowWidth`:

In [8]:
dataset.data_element('WindowWidth')

(0028,1051) Window Width                        DS: '4305'

We can visualize specific patient-related elements (`PatientID`, `PatientBirthDate`) to inspect values before anonymization

In [9]:
elements = ['PatientID',
                 'PatientBirthDate']
for element in elements:
    print(dataset.data_element(element))

(0010,0020) Patient ID                          LO: '1234'
(0010,0030) Patient's Birth Date                DA: '19981231'


We can define a callback function to find for example all tags corresponding to a person
names inside the dataset.

Two callback functions are defined in the following lines: `person_names_callback` replaces values whose VR is `PN` (person name), and `scanner_info_callback` replaces `LO` VR values (scanner info).

In [10]:
def person_names_callback(dataset, data_element):
    if data_element.VR == "PN": #VR = value representation, PN = persons's name
        data_element.value = "anonymous"

def scanner_info_callback(dataset, data_element):
    if data_element.VR == "LO":
        data_element.value = "scanner info"

We can use the different callback function to iterate through the dataset but
also some other tags such that patient ID, etc.

This can be achieved by means of the `walk` method, which iterates over the data elements of the *FileDataset* object:

In [11]:
dataset.walk(person_names_callback)
dataset.walk(scanner_info_callback)

or, equivalently, as:

In [12]:
callbacks = [person_names_callback, scanner_info_callback]
for callback in callbacks:
    dataset.walk(callback)

pydicom allows to remove private tags using `remove_private_tags` method

In [13]:
dataset.remove_private_tags()

Optional data elements can be easily deleted using `del` or `delattr`.



In [14]:
if 'OtherPatientIDs' in dataset:
    delattr(dataset, 'OtherPatientIDs')

if 'OtherPatientIDsSequence' in dataset:
    del dataset.OtherPatientIDsSequence

Data can also be modified via assignments:

In [15]:
dataset.OperatorsName= 'Lucio Verdi'


## Anonymize/modify other data

This section shows parsing a DICOM date string, creating a `datetime` object, replacing day/month via `replace(...)`, formatting it back to `YYYYMMDD`, and updating `PatientBirthDate`.

Let's try to set the birth date to the first day of the month.

In [ ]:
import datetime
import time

In [ ]:
date = '20000122'

In [ ]:
format_ = "%Y%m%d"
time_struct = time.strptime(date, format_)
time_struct

time.struct_time(tm_year=2000, tm_mon=1, tm_mday=22, tm_hour=0, tm_min=0, tm_sec=0, tm_wday=5, tm_yday=22, tm_isdst=-1)

In [ ]:
birth_date = datetime.datetime(*time_struct[:3])
birth_date

datetime.datetime(2000, 1, 22, 0, 0)

datetime.datetime objects are immutable

In [ ]:
#birth_date.day = 1

In [ ]:
new_date = birth_date.replace(day=1, month=5)
new_date

datetime.datetime(2000, 5, 1, 0, 0)

In [ ]:
new_date.strftime(format_)

'20000501'

An `anonimize_day` function is provided to consistently set the day component when anonymizing dates.

In [ ]:
def anonimize_day(date_str, format_="%Y%m%d", day=1):
    time_struct = time.strptime(date_str, format_)
    date = datetime.datetime(*time_struct[:3])
    new_date = date.replace(day=day)
    return new_date.strftime(format_)


In [ ]:
tag = 'PatientBirthDate'
if tag in dataset:
    date_str = dataset.data_element(tag).value
    dataset.data_element(tag).value = anonimize_day(date_str, day=5)
dataset.PatientBirthDate

'19981205'

## Store the anonymized DICOM file image

Finally, it is possible to write the anonymized DICOM file in a new file



In [16]:
output_filename = os.path.join(DATASETS, "IMAGES", "DICOM_Examples", "Brain_MRI", "IM67_1slice_anon.dcm")
dataset.save_as(output_filename)